# WT vs one all-lysine-to-arginine mutant

This Colab notebook performs one focused comparison:

1. Clone the therapeutic-optimization repository.
2. Accept one wild-type sequence and a list of lysine positions.
3. Make **one mutant** in which all listed lysines are changed to arginine simultaneously.
4. Predict only the WT and that one mutant with ColabFold.
5. Calculate structural-change metrics and generate a per-residue displacement graph.

No EUP run, individual mutants, combinatorial mutants, ranking, or ubiquitination rescoring is performed. Use a **GPU runtime** in Colab.

## 1. Clone the repository and install the minimal dependencies

The first ColabFold run may download model weights and can take some time.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/juliaevizza/Therapeutic_optimization.git"
REPO_DIR = Path("/content/therapeutic_optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )
elif (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")

# Install this project without the optional EUP dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

# Put the real src package ahead of /content. This prevents the outer clone
# directory from being mistaken for an empty namespace package.
SRC_DIR = REPO_DIR / "src"
sys.path.insert(0, str(SRC_DIR))
os.chdir(REPO_DIR)
for module_name in list(sys.modules):
    if module_name == "therapeutic_optimization" or module_name.startswith("therapeutic_optimization."):
        del sys.modules[module_name]

# Install ColabFold only if this runtime does not already provide it.
if shutil.which("colabfold_batch") is None:
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "colabfold[alphafold,openmm]", "jax[cuda12]", "openmm[cuda12]",
        ],
        check=True,
    )

print("Repository:", REPO_DIR)
print("ColabFold:", shutil.which("colabfold_batch"))
subprocess.run(["nvidia-smi"], check=False)

## 2. Enter the WT sequence and lysine positions

Paste the lysine positions from your UP1 result into `LYSINE_POSITIONS`. Positions are 1-based. Every listed change is combined into one mutant. The validation cell also requires the list to contain every lysine in the supplied WT sequence.

In [ ]:
PROTEIN_ID = "my_protein"

WT_SEQUENCE = """PTAPPYDSLLVFDYEGGSGSGSGASRLNFGDDIPSALRIAKKKRWNSIEERRIHQESELHSYLSRLIAAERERELEECQRNHEGDEDDSHVRAQQACIEAKHDKYMADMDELFSQVDEKRKKRDIPDYLCGKISFELMREPCITPSGITYDRKDIEEHLQRVGHFDPVTRSPLTQEQLIPNLAMKEVIDAFISENGWVEDY"""

# Paste the 1-based lysine positions here. They are all applied together.
LYSINE_POSITIONS = [41, 42, 43, 101, 104, 119, 121, 122, 132, 153, 185]

# One model per sequence is the quickest focused comparison.
NUM_MODELS = 1
NUM_RECYCLES = 3

## 3. Validate the inputs and create exactly two FASTA files

This creates a WT FASTA and one combined `ALL_K_TO_R` mutant FASTA. It does not create single or combinatorial mutant libraries.

In [ ]:
import pandas as pd
from therapeutic_optimization.io import (
    apply_mutations,
    normalize_sequence,
    write_fasta,
)

RUN_ROOT = Path("/content/wt_vs_all_lysines")
FASTA_DIR = RUN_ROOT / "fastas"
STRUCTURE_DIR = RUN_ROOT / "structures"
TABLE_DIR = RUN_ROOT / "tables"
PER_RESIDUE_DIR = RUN_ROOT / "per_residue"
FIGURE_DIR = RUN_ROOT / "figures"
for directory in (FASTA_DIR, STRUCTURE_DIR, TABLE_DIR, PER_RESIDUE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

wt_sequence = normalize_sequence(WT_SEQUENCE)
wt_lysine_positions = [
    position for position, residue in enumerate(wt_sequence, start=1) if residue == "K"
]
requested_positions = sorted(set(int(position) for position in LYSINE_POSITIONS))

if requested_positions != wt_lysine_positions:
    missing = sorted(set(wt_lysine_positions) - set(requested_positions))
    unexpected = sorted(set(requested_positions) - set(wt_lysine_positions))
    raise ValueError(
        "LYSINE_POSITIONS must contain every lysine in the WT sequence exactly once. "
        f"Missing WT lysines: {missing}; non-lysine/unexpected positions: {unexpected}"
    )

mutations = [f"K{position}R" for position in requested_positions]
mutation_spec = ";".join(mutations)
mutant_sequence = apply_mutations(wt_sequence, mutations)

WT_FASTA = write_fasta(f"{PROTEIN_ID}_WT", wt_sequence, FASTA_DIR / "WT.fasta")
MUTANT_FASTA = write_fasta(
    f"{PROTEIN_ID}_ALL_K_TO_R", mutant_sequence, FASTA_DIR / "ALL_K_TO_R.fasta"
)

print("Sequence length:", len(wt_sequence))
print("Combined mutations:", mutation_spec)
print("WT FASTA:", WT_FASTA)
print("Mutant FASTA:", MUTANT_FASTA)

## 4. Predict the WT and the one combined mutant

Both sequences are submitted in one ColabFold batch. Only one model is requested for each sequence by default.

In [ ]:
from therapeutic_optimization.structural_analysis.colabfold import (
    ColabFoldPredictor,
    StructurePrediction,
)

predictor = ColabFoldPredictor(
    executable="colabfold_batch",
    extra_args=(
        "--num-models", str(NUM_MODELS),
        "--num-recycle", str(NUM_RECYCLES),
    ),
)

predicted_structures = predictor.predict_batch(
    [
        StructurePrediction("WT", WT_FASTA, STRUCTURE_DIR / "WT"),
        StructurePrediction("ALL_K_TO_R", MUTANT_FASTA, STRUCTURE_DIR / "ALL_K_TO_R"),
    ],
    batch_output_dir=STRUCTURE_DIR / "colabfold_batch",
)

WT_STRUCTURE = predicted_structures["WT"]
MUTANT_STRUCTURE = predicted_structures["ALL_K_TO_R"]
print("WT structure:", WT_STRUCTURE)
print("Mutant structure:", MUTANT_STRUCTURE)

## 5. Calculate and display the structural changes

In [ ]:
from IPython.display import Image, display
from therapeutic_optimization.structural_analysis.metrics import analyze_structure_pair

metrics = analyze_structure_pair(
    wt_structure_path=WT_STRUCTURE,
    mutant_structure_path=MUTANT_STRUCTURE,
    variant_id="ALL_K_TO_R",
    mutation_spec=mutation_spec,
    per_residue_dir=PER_RESIDUE_DIR,
    figure_dir=FIGURE_DIR,
)

metrics_table = pd.DataFrame([metrics])
METRICS_CSV = TABLE_DIR / "WT_vs_ALL_K_TO_R_structural_metrics.csv"
metrics_table.to_csv(METRICS_CSV, index=False)

summary_metrics = [
    "global_ca_rmsd",
    "mean_ca_displacement",
    "max_ca_displacement",
    "mutation_ca_displacement",
    "mutation_ca_displacement_max",
    "local_mean_ca_displacement",
    "radius_of_gyration_change",
    "mean_plddt_change",
    "global_contact_change_fraction",
    "local_contacts_lost",
    "local_contacts_gained",
]
display(metrics_table[summary_metrics].T.rename(columns={0: "value"}))
display(Image(filename=metrics["displacement_plot"]))
print("Full metrics table:", METRICS_CSV)
print("Per-residue data:", metrics["per_residue_csv"])
print("Graph:", metrics["displacement_plot"])

## 6. Download all results

This downloads one ZIP containing the FASTAs, predicted structures, metrics table, per-residue data, graph, and ColabFold log.

In [ ]:
RESULTS_ZIP = Path(
    shutil.make_archive(
        "/content/WT_vs_ALL_K_TO_R_structural_results",
        "zip",
        root_dir=RUN_ROOT,
    )
)
print("Results archive:", RESULTS_ZIP)

try:
    from google.colab import files
    files.download(str(RESULTS_ZIP))
except ImportError:
    print("Automatic download is available when this notebook runs in Google Colab.")